# 해시

### 1. 폰켓몬

In [ ]:
def solution(nums):
    result = dict()
    for i in set(nums):
        result[i] = 0
    for j in nums:
        result[j] += 1
    if nums/2 > len(result):
        return len(result)
    else:
        return nums/2

### 2. 완주하지 못한 선수

In [20]:
#시간초과
def solution(participant, completion):
    for i in completion:
        participant.remove(i)
        return participant[0]
    
#통과
from collections import defaultdict
def solution(participant, completion):
    result = defaultdict(int)
    for i in participant:
        result[i] +=1
    for j in completion:
        result[j] -= 1
    for k in result:
        if result[k] == 1:
            return k

defaultdict(<class 'int'>, {'marina': 0, 'josipa': 0, 'nikola': 0, 'vinko': 1, 'filipa': 0})


'vinko'

In [ ]:
import json
import pandas as pd
import re
import emoji
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
import nltk
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import gensim
from gensim.models.coherencemodel import CoherenceModel
from nltk.stem import PorterStemmer  # Stemming 추가

# nltk 리소스 다운로드 (불용어 제거에 사용)
# nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# MiniLM 모델 로드 (paraphrase-MiniLM-L6-v2)
model_name = 'paraphrase-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

# JSON 파일 로드
json_file_path = 'Tower_of_Fantasy_970.json'
with open(json_file_path, 'r') as file:
    data = json.load(file)

# JSON 데이터에서 추천 리뷰 추출
extracted_texts = []
for item in data.values():
    if isinstance(item, list) and len(item) > 2 : #and item[1] == 'Recommended':
        extracted_texts.append(item[2]) #.replace("game", '')

# 전처리 함수 (이모지, 특수문자 제거, 불용어 제거, Stemming)
def preprocess_text(text):
    text = emoji.replace_emoji(text, replace='')  # 이모지 제거
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # 특수문자 제거
    words = text.split()
    stemmer = PorterStemmer()
    filtered_words = [stemmer.stem(word) for word in words if word.lower() not in stop_words ]  #and word.lower() not in ('game','games')
    return ' '.join(filtered_words)

# 전처리된 텍스트 저장
preprocessed_texts = [preprocess_text(text) for text in extracted_texts]

# DataFrame 생성
df = pd.DataFrame({
    'text': extracted_texts,
    'preprocessed_text': preprocessed_texts,
})

# LDA 모델 학습을 위한 품사 필터링 (명사 & 형용사만 남김)
nlp = spacy.load("en_core_web_sm")
def extract_noun_adj(text):
    doc = nlp(text)
    return ' '.join([token.text for token in doc if token.pos_ in ['NOUN', 'ADJ']])

df['preprocessed_text'] = df['preprocessed_text'].apply(extract_noun_adj)

# 벡터화 (n-gram 사용 X)
vectorizer = CountVectorizer(max_df=0.9, min_df=2, stop_words='english')
X = vectorizer.fit_transform(df['preprocessed_text'])

n_coms = [6]
for num_com in n_coms:
    # LDA 모델 학습
    lda = LatentDirichletAllocation(n_components=num_com, random_state=42)
    lda.fit(X)

    # Perplexity 확인
    perplexity = lda.perplexity(X)
    #print(f"Perplexity: {perplexity}")

    # 토픽 출력 함수
    def display_topics(model, feature_names, no_top_words):
        for topic_idx, topic in enumerate(model.components_):
            print(f"Topic {topic_idx + 1}:")
            print(", ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))
            print()

    display_topics(lda, vectorizer.get_feature_names_out(), 10)

Topic 1:
genshin, game, good, impact, better, fun, play, nice, friend, best

Topic 2:
game, player, weapon, content, new, server, time, money, play, charact

Topic 3:
game, charact, stori, explor, player, new, combat, time, world, genshin

Topic 4:
game, account, product, receiv, free, time, steam, delet, good, charact

Topic 5:
game, time, good, player, lot, weapon, thing, charact, play, fun

Topic 6:
game, fun, great, charact, time, good, control, lot, stori, bug

